# Corrigé du TP bonus C : PySpark, le même travail qu'avec pandas

## Objectifs

En une quarantaine de minutes, refaire avec PySpark des opérations déjà faites avec pandas, et comparer les deux écritures :

- lire une table et l'explorer ;
- comprendre la différence entre transformation et action (l'évaluation paresseuse) ;
- ajouter des colonnes, filtrer, agréger, joindre ;
- écrire la même requête en SQL ;
- revenir à pandas, et mesurer qui va le plus vite sur nos données.

## Les données

- `ventes_consolidees.parquet` : la table des ventes du TP3, une ligne par commande
- `regions.csv` : les six régions et leur objectif mensuel

## Comment travailler

- Complète les cellules marquées `# TODO`, en utilisant les noms de variables indiqués.
- Chaque question rappelle l'écriture pandas, pour comparer.
- Les cellules `# Vérification` comparent ton résultat PySpark à celui de pandas avec `assert` : exécute-les sans les modifier. Si tout est juste, elles affichent `OK`.
- La question **Pour aller plus loin** est facultative.

## Partie 0 : environnement

PySpark s'appuie sur Spark, écrit en Scala, qui tourne dans une machine virtuelle Java. Il faut donc Java 17 ou plus récent. Dans un terminal, vérifie :

```bash
java -version
```

Si la commande est introuvable, installe Java :

```bash
sudo apt install openjdk-17-jre-headless
```

Puis, dans l'environnement virtuel des TP précédents :

```bash
python -m pip install pyspark
```

Lance `jupyter lab` depuis le dossier de ce TP et exécute la cellule suivante. Elle démarre une session Spark locale, qui utilise tous les cœurs de ta machine, et charge aussi les ventes avec pandas pour comparer. Le démarrage prend quelques secondes et affiche des avertissements en rouge : c'est normal.

In [ ]:
import time

import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder.master("local[*]")
    .appName("tp-bonus-pyspark")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

ventes_pd = pd.read_parquet("ressources/ventes_consolidees.parquet")
print("Spark", spark.version, "| pandas", pd.__version__)


## Partie 1 : lire et explorer

`spark.read` lit les formats courants : `spark.read.parquet(chemin)`, `spark.read.csv(chemin, header=True, inferSchema=True)`, `spark.read.json(chemin)`. Le résultat est un DataFrame Spark : il ressemble à un DataFrame pandas, mais ses données ne sont pas en mémoire dans Python.

| pandas | PySpark |
|---|---|
| `df.head()` | `df.show(5)` |
| `df.dtypes`, `df.info()` | `df.printSchema()` |
| `len(df)` | `df.count()` |
| `df.columns` | `df.columns` |

### Question 1.1

Lis `ressources/ventes_consolidees.parquet` avec Spark dans une variable `ventes`, puis affiche son schéma avec `printSchema()`. Les types sont-ils les mêmes qu'avec pandas ?

In [2]:
ventes = spark.read.parquet("ressources/ventes_consolidees.parquet")
ventes.printSchema()


root
 |-- id_commande: long (nullable = true)
 |-- date: timestamp_ntz (nullable = true)
 |-- code_region: string (nullable = true)
 |-- id_produit: string (nullable = true)
 |-- quantite: long (nullable = true)
 |-- prix_unitaire: double (nullable = true)
 |-- client_fidele: boolean (nullable = true)
 |-- fichier: string (nullable = true)
 |-- produit: string (nullable = true)
 |-- categorie: string (nullable = true)
 |-- prix_catalogue: double (nullable = true)
 |-- region: string (nullable = true)
 |-- objectif_mensuel: long (nullable = true)
 |-- montant: double (nullable = true)
 |-- jour: string (nullable = true)



In [3]:
# Vérification
from pyspark.sql import DataFrame

assert isinstance(ventes, DataFrame), "ventes doit être un DataFrame Spark : spark.read.parquet(...)"
assert ventes.count() == len(ventes_pd), "ventes doit contenir toutes les commandes"
print("Question 1.1 : OK")


Question 1.1 : OK


**Réponse** : Oui, aux noms près : `long` pour les entiers (`int64`), `double` pour les décimaux (`float64`), `string` pour le texte, `timestamp_ntz` pour les dates sans fuseau horaire. Le format Parquet stocke les types avec les données, Spark n'a donc rien à deviner.

### Question 1.2

Crée `apercu`, qui ne garde que les colonnes `date`, `region`, `categorie` et `montant` avec `select`, puis affiche ses 5 premières lignes avec `apercu.show(5)`.

En pandas : `ventes_pd[["date", "region", "categorie", "montant"]].head(5)`.

**Résultat attendu** :

```text
+-------------------+------------------+--------------+------------------+
|               date|            region|     categorie|           montant|
+-------------------+------------------+--------------+------------------+
|2025-01-01 00:00:00|     Île-de-France|  Informatique|             84.55|
|2025-01-01 00:00:00|          Bretagne|     Papeterie|              88.2|
|2025-01-01 00:00:00|   Hauts-de-France|     Papeterie|16.599999999999998|
|2025-01-01 00:00:00|Nouvelle-Aquitaine|        Bureau|             224.1|
|2025-01-01 00:00:00|         Occitanie|Électroménager|209.70000000000002|
+-------------------+------------------+--------------+------------------+
only showing top 5 rows
```

In [4]:
apercu = ventes.select("date", "region", "categorie", "montant")
apercu.show(5)


+-------------------+------------------+--------------+------------------+
|               date|            region|     categorie|           montant|
+-------------------+------------------+--------------+------------------+
|2025-01-01 00:00:00|     Île-de-France|  Informatique|             84.55|
|2025-01-01 00:00:00|          Bretagne|     Papeterie|              88.2|
|2025-01-01 00:00:00|   Hauts-de-France|     Papeterie|16.599999999999998|
|2025-01-01 00:00:00|Nouvelle-Aquitaine|        Bureau|             224.1|
|2025-01-01 00:00:00|         Occitanie|Électroménager|209.70000000000002|
+-------------------+------------------+--------------+------------------+
only showing top 5 rows


In [5]:
# Vérification
assert apercu.columns == ["date", "region", "categorie", "montant"], "garde les quatre colonnes, dans cet ordre"
print("Question 1.2 : OK")


Question 1.2 : OK


### Question 1.3

Exécute la cellule suivante, qui garde les commandes de plus de 500 euros. Elle répond instantanément, et n'affiche rien. Complète-la pour compter ces commandes dans `nb_grosses` avec `count()`, puis affiche le résultat. Pourquoi la première ligne est-elle instantanée ?

En pandas : `len(ventes_pd[ventes_pd["montant"] > 500])`.

**Résultat attendu** :

```text
153
```

In [6]:
grosses = ventes.filter(F.col("montant") > 500)

nb_grosses = grosses.count()
print(nb_grosses)


153


In [7]:
# Vérification
assert nb_grosses == (ventes_pd["montant"] > 500).sum(), "compte les lignes de grosses avec grosses.count()"
print("Question 1.3 : OK")


Question 1.3 : OK


**Réponse** : Spark est paresseux : `filter`, `select` ou `withColumn` sont des **transformations**, qui ne font que noter ce qu'il faudra faire. Rien n'est calculé tant qu'on n'appelle pas une **action** : `count()`, `show()`, `collect()`, `toPandas()` ou une écriture. Spark connaît alors toute la chaîne d'opérations et peut l'optimiser avant de lancer le calcul.

## Partie 2 : transformer

Une colonne se désigne avec `F.col("nom")`, et le module `functions` (importé sous le nom `F`) fournit les fonctions : `F.month`, `F.year`, `F.round`, `F.upper`, `F.when`... Un DataFrame Spark ne se modifie jamais : chaque opération renvoie un nouveau DataFrame, qu'on range dans une variable.

| pandas | PySpark |
|---|---|
| `df["b"] = df["a"] * 2` | `df = df.withColumn("b", F.col("a") * 2)` |
| `df[df["a"] > 0]` | `df.filter(F.col("a") > 0)` |
| `(cond1) & (cond2)` | `(cond1) & (cond2)` |
| `df.sort_values("a", ascending=False)` | `df.orderBy(F.desc("a"))` |

### Question 2.1

À partir de `ventes`, crée `ventes2` avec deux nouvelles colonnes :

- `mois` : le numéro du mois de la commande, avec `F.month("date")` ;
- `montant_ttc` : le montant multiplié par 1,2, arrondi à 2 décimales avec `F.round(..., 2)`.

Enchaîne les deux `withColumn`, puis affiche 3 lignes des colonnes `date`, `mois`, `montant` et `montant_ttc`.

En pandas : `ventes_pd["mois"] = ventes_pd["date"].dt.month` et `ventes_pd["montant_ttc"] = (ventes_pd["montant"] * 1.2).round(2)`.

**Résultat attendu** :

```text
+-------------------+----+------------------+-----------+
|               date|mois|           montant|montant_ttc|
+-------------------+----+------------------+-----------+
|2025-01-01 00:00:00|   1|             84.55|     101.46|
|2025-01-01 00:00:00|   1|              88.2|     105.84|
|2025-01-01 00:00:00|   1|16.599999999999998|      19.92|
+-------------------+----+------------------+-----------+
only showing top 3 rows
```

In [8]:
ventes2 = (
    ventes.withColumn("mois", F.month("date"))
    .withColumn("montant_ttc", F.round(F.col("montant") * 1.2, 2))
)
ventes2.select("date", "mois", "montant", "montant_ttc").show(3)


+-------------------+----+------------------+-----------+
|               date|mois|           montant|montant_ttc|
+-------------------+----+------------------+-----------+
|2025-01-01 00:00:00|   1|             84.55|     101.46|
|2025-01-01 00:00:00|   1|              88.2|     105.84|
|2025-01-01 00:00:00|   1|16.599999999999998|      19.92|
+-------------------+----+------------------+-----------+
only showing top 3 rows


In [9]:
# Vérification
assert {"mois", "montant_ttc"} <= set(ventes2.columns), "ventes2 doit avoir les colonnes mois et montant_ttc"
assert "mois" not in ventes.columns, "ventes ne change pas : withColumn renvoie un nouveau DataFrame"
mois_spark = sorted(ligne["mois"] for ligne in ventes2.select("mois").distinct().collect())
assert mois_spark == list(range(1, 10)), "mois doit aller de 1 à 9 : F.month(\"date\")"
total_ttc = ventes2.agg(F.sum("montant_ttc")).first()[0]
assert abs(total_ttc - (ventes_pd["montant"] * 1.2).round(2).sum()) < 0.01, "montant_ttc : montant * 1.2, arrondi à 2 décimales"
print("Question 2.1 : OK")


Question 2.1 : OK


### Question 2.2

Dans `fideles_bretagne`, garde les commandes de clients fidèles (`client_fidele` vrai) passées en Bretagne, triées par montant décroissant. Affiche les 3 premières avec les colonnes `date`, `produit` et `montant`.

En pandas : `ventes_pd[(ventes_pd["region"] == "Bretagne") & ventes_pd["client_fidele"]].sort_values("montant", ascending=False)`.

**Résultat attendu** :

```text
+-------------------+-------------------+-------+
|               date|            produit|montant|
+-------------------+-------------------+-------+
|2025-02-24 00:00:00|Bureau assis-debout| 1596.0|
|2025-04-09 00:00:00| Chaise ergonomique| 1245.0|
|2025-01-17 00:00:00| Chaise ergonomique|  946.2|
+-------------------+-------------------+-------+
only showing top 3 rows
```

In [10]:
fideles_bretagne = ventes2.filter(
    (F.col("region") == "Bretagne") & F.col("client_fidele")
).orderBy(F.desc("montant"))
fideles_bretagne.select("date", "produit", "montant").show(3)


+-------------------+-------------------+-------+
|               date|            produit|montant|
+-------------------+-------------------+-------+
|2025-02-24 00:00:00|Bureau assis-debout| 1596.0|
|2025-04-09 00:00:00| Chaise ergonomique| 1245.0|
|2025-01-17 00:00:00| Chaise ergonomique|  946.2|
+-------------------+-------------------+-------+
only showing top 3 rows


In [11]:
# Vérification
attendu = ventes_pd[(ventes_pd["region"] == "Bretagne") & ventes_pd["client_fidele"]]
assert fideles_bretagne.count() == len(attendu), "garde les clients fidèles de Bretagne : (condition1) & (condition2)"
assert fideles_bretagne.first()["montant"] == attendu["montant"].max(), "trie par montant décroissant : orderBy(F.desc(\"montant\"))"
print("Question 2.2 : OK")


Question 2.2 : OK


## Partie 3 : agréger et joindre

`groupBy` puis `agg` regroupe et calcule plusieurs indicateurs à la fois. `alias` nomme chaque résultat. `join` remplace `merge`.

| pandas | PySpark |
|---|---|
| `df.groupby("g").agg(ca=("m", "sum"))` | `df.groupBy("g").agg(F.sum("m").alias("ca"))` |
| `("m", "count")`, `("m", "mean")` | `F.count("*")`, `F.avg("m")` |
| `df.merge(ref, on="cle")` | `df.join(ref, on="cle")` |
| `df.merge(ref, on="cle", how="left")` | `df.join(ref, on="cle", how="left")` |

### Question 3.1

Calcule dans `ca_region`, pour chaque région, le chiffre d'affaires arrondi à 2 décimales (`ca`) et le nombre de commandes (`nb_commandes`), trié par chiffre d'affaires décroissant. Affiche le résultat.

En pandas : `ventes_pd.groupby("region").agg(ca=("montant", "sum"), nb_commandes=("montant", "count")).sort_values("ca", ascending=False)`.

**Résultat attendu** :

```text
+--------------------+---------+------------+
|              region|       ca|nb_commandes|
+--------------------+---------+------------+
|       Île-de-France|127936.94|         674|
|  Nouvelle-Aquitaine| 71843.18|         313|
|Auvergne-Rhône-Alpes| 64856.25|         378|
|     Hauts-de-France| 53467.33|         320|
|           Occitanie| 50318.18|         288|
|            Bretagne| 41079.71|         209|
+--------------------+---------+------------+
```

In [12]:
ca_region = (
    ventes2.groupBy("region")
    .agg(
        F.round(F.sum("montant"), 2).alias("ca"),
        F.count("*").alias("nb_commandes"),
    )
    .orderBy(F.desc("ca"))
)
ca_region.show()


+--------------------+---------+------------+
|              region|       ca|nb_commandes|
+--------------------+---------+------------+
|       Île-de-France|127936.94|         674|
|  Nouvelle-Aquitaine| 71843.18|         313|
|Auvergne-Rhône-Alpes| 64856.25|         378|
|     Hauts-de-France| 53467.33|         320|
|           Occitanie| 50318.18|         288|
|            Bretagne| 41079.71|         209|
+--------------------+---------+------------+



In [13]:
# Vérification
attendu = ventes_pd.groupby("region")["montant"].agg(["sum", "count"]).sort_values("sum", ascending=False)
lignes = ca_region.collect()
assert [ligne["region"] for ligne in lignes] == list(attendu.index), "trie les régions par chiffre d'affaires décroissant"
assert all(abs(ligne["ca"] - attendu.loc[ligne["region"], "sum"]) < 0.01 for ligne in lignes), "ca : somme des montants, arrondie"
assert all(ligne["nb_commandes"] == attendu.loc[ligne["region"], "count"] for ligne in lignes), "nb_commandes : F.count(\"*\")"
print("Question 3.1 : OK")


Question 3.1 : OK


### Question 3.2

Lis `ressources/regions.csv` avec Spark dans `regions` (`header=True, inferSchema=True`). Puis construis `taux` :

1. le chiffre d'affaires arrondi à 2 décimales (`ca`) de chaque couple `code_region` et `mois`, à partir de `ventes2` ;
2. joint à `regions` sur `code_region`, pour récupérer `region` et `objectif_mensuel` ;
3. avec une colonne `taux_realisation` égale à `ca / objectif_mensuel`, arrondie à 2 décimales.

Affiche les lignes du mois d'août (`mois == 8`), avec les colonnes `region`, `ca` et `taux_realisation`. Combien de régions atteignent leur objectif ?

**Résultat attendu** :

```text
+--------------------+-------+----------------+
|              region|     ca|taux_realisation|
+--------------------+-------+----------------+
|Auvergne-Rhône-Alpes|2080.84|            0.24|
|            Bretagne| 1361.0|             0.3|
|     Hauts-de-France|3363.03|            0.52|
|  Nouvelle-Aquitaine|6483.44|            0.93|
|           Occitanie|3998.31|            0.62|
|       Île-de-France|9987.01|            0.71|
+--------------------+-------+----------------+
```

In [14]:
regions = spark.read.csv("ressources/regions.csv", header=True, inferSchema=True)

taux = (
    ventes2.groupBy("code_region", "mois")
    .agg(F.round(F.sum("montant"), 2).alias("ca"))
    .join(regions, on="code_region")
    .withColumn("taux_realisation", F.round(F.col("ca") / F.col("objectif_mensuel"), 2))
)
taux.filter(F.col("mois") == 8).select("region", "ca", "taux_realisation").orderBy("region").show()


+--------------------+-------+----------------+
|              region|     ca|taux_realisation|
+--------------------+-------+----------------+
|Auvergne-Rhône-Alpes|2080.84|            0.24|
|            Bretagne| 1361.0|             0.3|
|     Hauts-de-France|3363.03|            0.52|
|  Nouvelle-Aquitaine|6483.44|            0.93|
|           Occitanie|3998.31|            0.62|
|       Île-de-France|9987.01|            0.71|
+--------------------+-------+----------------+



In [15]:
# Vérification
assert taux.count() == 54, "il faut une ligne par région et par mois : 6 x 9"
assert {"region", "objectif_mensuel", "taux_realisation"} <= set(taux.columns), "joins regions sur code_region, puis ajoute taux_realisation"
assert taux.filter((F.col("mois") == 8) & (F.col("taux_realisation") >= 1)).count() == 0, "vérifie le calcul de taux_realisation"
print("Question 3.2 : OK")


Question 3.2 : OK


**Réponse** : Aucune : en août, toutes les régions sont sous leur objectif. On retrouve l'effet saisonnier vu sur la heatmap du TP4.

## Partie 4 : le même calcul en SQL

Spark comprend aussi le SQL. On donne d'abord un nom de table au DataFrame avec `createOrReplaceTempView`, puis `spark.sql` renvoie un DataFrame, qu'on manipule comme les autres. Les deux écritures produisent exactement le même plan d'exécution : on choisit la plus lisible.

```python
ventes2.createOrReplaceTempView("ventes")
spark.sql("SELECT region, COUNT(*) AS n FROM ventes GROUP BY region").show()
```

### Question 4.1

Enregistre `ventes2` comme vue `ventes`, puis calcule en SQL dans `ca_categorie` le chiffre d'affaires de chaque catégorie, arrondi à 2 décimales (`ROUND(SUM(montant), 2) AS ca`), trié par chiffre d'affaires décroissant. Affiche le résultat.

**Résultat attendu** :

```text
+--------------+---------+
|     categorie|       ca|
+--------------+---------+
|  Informatique|185274.68|
|        Bureau|124548.45|
|     Papeterie|  59180.8|
|Électroménager| 37824.04|
| Non référencé|  2673.62|
+--------------+---------+
```

In [16]:
ventes2.createOrReplaceTempView("ventes")

ca_categorie = spark.sql("""
    SELECT categorie, ROUND(SUM(montant), 2) AS ca
    FROM ventes
    GROUP BY categorie
    ORDER BY ca DESC
""")
ca_categorie.show()


+--------------+---------+
|     categorie|       ca|
+--------------+---------+
|  Informatique|185274.68|
|        Bureau|124548.45|
|     Papeterie|  59180.8|
|Électroménager| 37824.04|
| Non référencé|  2673.62|
+--------------+---------+



In [17]:
# Vérification
attendu = ventes_pd.groupby("categorie")["montant"].sum().sort_values(ascending=False)
lignes = ca_categorie.collect()
assert [ligne["categorie"] for ligne in lignes] == list(attendu.index), "trie les catégories par chiffre d'affaires décroissant"
assert all(abs(ligne["ca"] - attendu[ligne["categorie"]]) < 0.01 for ligne in lignes), "ca : ROUND(SUM(montant), 2)"
print("Question 4.1 : OK")


Question 4.1 : OK


## Partie 5 : retour à pandas, et qui va le plus vite ?

`toPandas()` rapatrie un DataFrame Spark dans un DataFrame pandas, en mémoire. On l'utilise sur un résultat **agrégé**, petit, par exemple pour tracer un graphique : sur une table de plusieurs milliards de lignes, il ferait déborder la mémoire.

### Question 5.1

Convertis `ca_region` en DataFrame pandas dans `ca_region_pd`, et affiche-le. Un avertissement `FutureWarning` sur la version de pandas peut s'afficher : il est sans conséquence ici.

In [18]:
ca_region_pd = ca_region.toPandas()
ca_region_pd


/private/tmp/claude-501/-Users-arnaudrousseau-Documents-projets-personnal-formation-formation-analyse-donnees-python/d5e00574-7e54-4972-b1a5-77d306c11044/scratchpad/.venv/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


,region,ca,nb_commandes
0,Île-de-France,127936.94,674
1,Nouvelle-Aquitaine,71843.18,313
2,Auvergne-Rhône-Alpes,64856.25,378
3,Hauts-de-France,53467.33,320
4,Occitanie,50318.18,288
5,Bretagne,41079.71,209


In [19]:
# Vérification
assert isinstance(ca_region_pd, pd.DataFrame), "ca_region_pd doit être un DataFrame pandas : toPandas()"
assert len(ca_region_pd) == 6, "il faut une ligne par région"
print("Question 5.1 : OK")


Question 5.1 : OK


### Question 5.2

Exécute la cellule suivante : elle mesure le temps du même calcul (chiffre d'affaires par région) avec pandas puis avec Spark. Lequel est le plus rapide, et pourquoi ? Dans quel cas Spark prendrait-il l'avantage ?

In [20]:
debut = time.perf_counter()
ventes_pd.groupby("region")["montant"].sum()
duree_pandas = time.perf_counter() - debut

debut = time.perf_counter()
ventes.groupBy("region").agg(F.sum("montant")).collect()
duree_spark = time.perf_counter() - debut

print(f"pandas : {duree_pandas * 1000:.1f} ms")
print(f"Spark  : {duree_spark * 1000:.1f} ms")


pandas : 0.6 ms
Spark  : 49.0 ms


**Réponse** : pandas gagne largement, avec moins d'une milliseconde contre plusieurs dizaines. Sur 2 182 lignes, Spark perd son temps à planifier le calcul, découper les données, lancer des tâches dans la machine Java et rapatrier le résultat. Spark prend l'avantage quand les données ne tiennent plus en mémoire sur une machine (des dizaines de gigaoctets et plus) : il les traite par morceaux, en parallèle, sur plusieurs cœurs ou sur un cluster de machines, sans changer le code.

### Pour aller plus loin 5.3

Spark écrit ses résultats dans un **dossier**, en plusieurs fichiers, et peut les ranger par valeur d'une colonne. Écris `ventes2` au format Parquet dans `sortie/ventes_par_mois`, partitionné par mois :

```python
ventes2.write.mode("overwrite").partitionBy("mois").parquet("sortie/ventes_par_mois")
```

Puis affiche la liste des sous-dossiers créés avec `sorted(p.name for p in Path("sortie/ventes_par_mois").iterdir() if not p.name.startswith("."))`. À quoi sert ce découpage ?

**Résultat attendu** :

```text
['_SUCCESS',
 'mois=1',
 'mois=2',
 'mois=3',
 'mois=4',
 'mois=5',
 'mois=6',
 'mois=7',
 'mois=8',
 'mois=9']
```

In [21]:
from pathlib import Path

ventes2.write.mode("overwrite").partitionBy("mois").parquet("sortie/ventes_par_mois")
sorted(p.name for p in Path("sortie/ventes_par_mois").iterdir() if not p.name.startswith("."))


['_SUCCESS',
 'mois=1',
 'mois=2',
 'mois=3',
 'mois=4',
 'mois=5',
 'mois=6',
 'mois=7',
 'mois=8',
 'mois=9']

In [22]:
# Vérification
from pathlib import Path

dossiers = [p for p in Path("sortie/ventes_par_mois").iterdir() if p.name.startswith("mois=")]
assert len(dossiers) == 9, "il faut un sous-dossier par mois : partitionBy(\"mois\")"
print("Question 5.3 : OK")


Question 5.3 : OK


**Réponse** : Chaque mois a son sous-dossier `mois=1`, `mois=2`... Une lecture filtrée, par exemple `spark.read.parquet("sortie/ventes_par_mois").filter(F.col("mois") == 8)`, n'ouvre que le dossier `mois=8` et ignore les autres. Sur de gros volumes, c'est le premier levier de performance. Le fichier `_SUCCESS` signale que l'écriture s'est terminée sans erreur.

## Pour finir

Arrête la session Spark pour libérer la mémoire de la machine Java.

In [23]:
spark.stop()
